In [1]:
!nvidia-smi

Sun Apr 19 10:18:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.74                 Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   39C    P8              3W /   73W |     491MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch

def check_cuda():
    print("=== PyTorch CUDA Diagnostic ===\n")

    # 1. Is CUDA available?
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")

    # 2. PyTorch build info
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA version (torch): {torch.version.cuda}")

    if not cuda_available:
        print("\n[FAIL] CUDA is not available.")
        return

    # 3. GPU details
    device_count = torch.cuda.device_count()
    print(f"\nNumber of GPUs: {device_count}")

    for i in range(device_count):
        print(f"\n--- GPU {i} ---")
        print(f"Name: {torch.cuda.get_device_name(i)}")
        print(f"Capability: {torch.cuda.get_device_capability(i)}")
        print(f"Memory Allocated: {torch.cuda.memory_allocated(i)} bytes")
        print(f"Memory Reserved: {torch.cuda.memory_reserved(i)} bytes")

    # 4. Actual computation test (this is the only *real* proof)
    try:
        device = torch.device("cuda:0")
        x = torch.rand(3, 3).to(device)
        y = torch.rand(3, 3).to(device)
        z = x @ y

        print("\n[OK] Tensor computation on GPU succeeded.")
        print(f"Result device: {z.device}")

    except Exception as e:
        print("\n[FAIL] CUDA exists but computation failed.")
        print(f"Error: {e}")


if __name__ == "__main__":
    check_cuda()

=== PyTorch CUDA Diagnostic ===

CUDA available: True
PyTorch version: 2.12.0.dev20260324+cu128
CUDA version (torch): 12.8

Number of GPUs: 1

--- GPU 0 ---
Name: NVIDIA GeForce RTX 5060 Laptop GPU
Capability: (12, 0)
Memory Allocated: 0 bytes
Memory Reserved: 0 bytes

[OK] Tensor computation on GPU succeeded.
Result device: cuda:0


## Dataset Unzip

In [6]:
from pathlib import Path
import zipfile

# Paths
ROOT = Path.cwd().parent.parent
DATA_DIR = ROOT / "0_Datasets"

DATASET_PATH_ZIP = DATA_DIR / "Male_and_female.zip"
EXTRACT_DIR = DATA_DIR / "Male_and_female"

In [7]:
# Unzip
with zipfile.ZipFile(DATASET_PATH_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)

Extracted to: c:\Users\lucas\Desktop\Trabajo Final de Grado\TFG PinMartinez\New folder (2) - Copy\NEW\0_Datasets\Male_and_female


## Roboflow Dataset Inspection

In [8]:
from pathlib import Path

# Path
ROOT = Path.cwd().parent.parent

DATA_DIR = ROOT / "0_Datasets"

DATASET_PATH = DATA_DIR / "Male_and_female"

In [10]:
DATASET_PATH_YAML = DATASET_PATH / "data.yaml"

with open(DATASET_PATH_YAML, 'r') as f:
    print(f.read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['Female', 'Male']

roboflow:
  workspace: mosquitos-4tgsz
  project: male_abd_female
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/mosquitos-4tgsz/male_abd_female/dataset/1


In [9]:
import os
os.listdir(DATASET_PATH)

['data.yaml',
 'README.dataset.txt',
 'README.roboflow.txt',
 'test',
 'train',
 'valid']

In [ ]:
import os
from collections import defaultdict

def analizar_dataset_yolo_roboflow(dataset_path):
    splits = ['train', 'valid', 'test']
    ext_img = ('.jpg', '.jpeg', '.png')

    total_imgs = 0
    split_counts = {}
    class_counts = defaultdict(int)
    all_classes = set()

    print(f"\n Analyzing dataset in: {dataset_path}\n")

    for split in splits:
        images_dir = os.path.join(dataset_path, split, "images")
        labels_dir = os.path.join(dataset_path, split, "labels")

        if not os.path.exists(images_dir):
            print(f" No se encontró la carpeta {images_dir}")
            continue

        images = [f for f in os.listdir(images_dir) if f.lower().endswith(ext_img)]
        split_counts[split] = len(images)
        total_imgs += len(images)

        for img in images:
            base_name = os.path.splitext(img)[0]
            label_file = base_name + '.txt'
            label_path = os.path.join(labels_dir, label_file)
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        cls_id = line.strip().split()[0]
                        class_counts[cls_id] += 1
                        all_classes.add(cls_id)

    print(" Dataset Statistics:")
    print(f"   - Total images: {total_imgs}")
    for split in splits:
        count = split_counts.get(split, 0)
        pct = (count / total_imgs * 100) if total_imgs > 0 else 0
        print(f"   - {split.capitalize():<6}: {count} images ({pct:.2f}%)")

    print("\n Classes found:")
    if all_classes:
        sorted_classes = sorted(all_classes, key=lambda x: int(x))
        for cls_id in sorted_classes:
            print(f"   - Class {cls_id}: {class_counts[cls_id]} images")
    else:
        print("   - No se encontraron clases (verifica los archivos .txt)")

analizar_dataset_yolo_roboflow(DATASET_PATH)

## Creation of the classification dataset

In [11]:
from pathlib import Path

# Path
ROOT = Path.cwd().parent.parent

DATA_DIR = ROOT / "0_Datasets"

DATASET_PATH = DATA_DIR / "Male_and_female"

In [12]:
import os
import cv2
from pathlib import Path

# =========================
# CONFIG
# =========================
DATASET_ROOT = DATASET_PATH         # your roboflow dataset
OUTPUT_ROOT  = DATA_DIR / "Male_and_female_cls"      # new classification dataset

SPLITS = ["train", "valid", "test"]
CLASS_NAMES = ["Female", "Male"]  # from your YAML

# =========================
# UTILS
# =========================
def yolo_to_pixel(box, img_w, img_h):
    """
    Convert YOLO format (cx, cy, w, h) normalized
    to pixel coordinates (x1, y1, x2, y2)
    """
    cx, cy, w, h = box

    x1 = int((cx - w/2) * img_w)
    y1 = int((cy - h/2) * img_h)
    x2 = int((cx + w/2) * img_w)
    y2 = int((cy + h/2) * img_h)

    return x1, y1, x2, y2


def clamp_bbox(x1, y1, x2, y2, img_w, img_h):
    """
    Ensure bbox is inside image
    """
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img_w - 1, x2)
    y2 = min(img_h - 1, y2)

    return x1, y1, x2, y2


# =========================
# MAIN PROCESS
# =========================
for split in SPLITS:
    img_dir = Path(DATASET_ROOT) / split / "images"
    lbl_dir = Path(DATASET_ROOT) / split / "labels"

    for class_name in CLASS_NAMES:
        os.makedirs(Path(OUTPUT_ROOT) / split / class_name, exist_ok=True)

    image_files = list(img_dir.glob("*.*"))

    print(f"Processing {split}: {len(image_files)} images")

    for img_path in image_files:
        label_path = lbl_dir / (img_path.stem + ".txt")

        if not label_path.exists():
            continue  # skip images without labels

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        h, w = img.shape[:2]

        with open(label_path, "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls_id = int(parts[0])
            box = list(map(float, parts[1:]))

            x1, y1, x2, y2 = yolo_to_pixel(box, w, h)
            x1, y1, x2, y2 = clamp_bbox(x1, y1, x2, y2, w, h)

            # Avoid invalid crops
            if x2 <= x1 or y2 <= y1:
                continue

            crop = img[y1:y2, x1:x2]

            class_name = CLASS_NAMES[cls_id]
            save_dir = Path(OUTPUT_ROOT) / split / class_name

            # unique filename (important when multiple objects per image)
            save_name = f"{img_path.stem}_{i}.jpg"
            save_path = save_dir / save_name

            cv2.imwrite(str(save_path), crop)

print("Done. Classification dataset created.")

Processing train: 2103 images
Processing valid: 233 images
Processing test: 235 images
Done. Classification dataset created.


In [14]:
import os
import cv2
from pathlib import Path

# =========================
# DATASET PATH
# =========================
ROOT = Path.cwd().parent.parent

DATA_DIR = ROOT / "0_Datasets"

DATASET_ROOT = DATA_DIR / "Male_and_female_cls"

# =========================
# CONFIG
# =========================
MIN_SIZE = 64

removed = 0

for split in ["train", "valid", "test"]:
    for class_dir in Path(DATASET_ROOT, split).iterdir():
        for img_path in list(class_dir.glob("*.*")):
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            h, w = img.shape[:2]

            if h < MIN_SIZE or w < MIN_SIZE:
                os.remove(img_path)
                removed += 1

print(f"Removed {removed} small images")

Removed 33 small images


## ONNX Convertion

In [ ]:
import torch
import torchvision.models as models

# =========================
# CONFIG
# =========================
MODEL_PATH = "best_model_fixed.pth"   
ONNX_PATH = "best_CLmodel_fixed.onnx"
IMAGE_SIZE = 160
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 2

# =========================
# LOAD MODEL (MATCH TRAINING!)
# =========================
def build_model():
    model = models.mobilenet_v3_large(weights=None)  # <-- FIXED

    in_features = model.classifier[3].in_features
    model.classifier[3] = torch.nn.Linear(in_features, NUM_CLASSES)

    return model

model = build_model()

# Load weights
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)

model.to(DEVICE)
model.eval()

print("Model loaded correctly.")

# =========================
# DUMMY INPUT
# =========================
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)

# =========================
# EXPORT ONNX
# =========================
torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "logits": {0: "batch_size"}
    },
    dynamo=False
)

print(f"ONNX model exported to: {ONNX_PATH}")

Model loaded correctly.


C:\Users\lucas\AppData\Local\Temp\ipykernel_56800\2633547954.py:44: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model exported to: best_CLmodel_fixed.onnx
